# 📊 Clase 2 — Carga y Exploración de un Dataset Financiero Real (R)

## Aplicaciones de Minería de Datos a Economía y Finanzas
### Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026

**Docente:** Dr. Darío Ezequiel Díaz · drdarioezequieldiaz@gmail.com

**Fecha:** Jueves 16 de abril de 2026

---

### Objetivos del notebook

1. **Cargar** el German Credit Dataset (Statlog) desde el repositorio UCI utilizando `readr`.
2. **Explorar** su estructura con `glimpse()`, `skim()` y funciones del `tidyverse`.
3. **Analizar** la distribución de la variable objetivo y el desbalanceo 70/30.
4. **Examinar** variables categóricas y numéricas con estadísticos descriptivos.
5. **Visualizar** distribuciones y relaciones bivariadas con `ggplot2`.
6. **Diagnosticar** calidad de datos: faltantes, outliers, cardinalidad.
7. **Construir** variables derivadas con sentido económico-financiero.

### Dataset: German Credit Data (Statlog)

| Característica | Detalle |
|:---|:---|
| **Fuente** | UCI ML Repository / OpenML |
| **Registros** | 1.000 solicitantes de crédito |
| **Variables** | 20 predictoras + 1 objetivo |
| **Variable objetivo** | `class`: good (700) / bad (300) |
| **Desbalanceo** | 70/30 — genuino, moderado |

> **Nota sobre el entorno:** Este notebook opera bajo el kernel R en Google Colab. Los paquetes se cargan desde el script de configuración almacenado en Google Drive. Si es la primera vez que ejecuta este notebook, siga las instrucciones de la Sección 0.


---
## 0. Configuración del entorno en Colab

**Paso previo (ejecutar en celda Python antes de cambiar el runtime a R):**

```python
from google.colab import drive
drive.mount('/content/drive')
```

Luego: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → R*

La siguiente celda carga los paquetes desde su script de configuración en Drive. Si no dispone del script, los paquetes se instalan individualmente.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 0. CONFIGURACIÓN DEL ENTORNO R
# ══════════════════════════════════════════════════════════════

# Intentar cargar el script de setup desde Drive
setup_path <- "/content/drive/MyDrive/R_Colab/setup_R_colab.R"

if (file.exists(setup_path)) {
  cat("✅ Cargando entorno desde setup_R_colab.R...\n")
  source(setup_path)
} else {
  cat("ℹ️  Script de setup no encontrado.\n")
}

# Paquetes requeridos para este notebook
paquetes_requeridos <- c("tidyverse", "skimr", "corrplot", "gridExtra",
                         "scales", "knitr")

# Verificar e instalar solo los que falten (post-setup)
faltantes <- paquetes_requeridos[!(paquetes_requeridos %in% installed.packages()[, "Package"])]
if (length(faltantes) > 0) {
  cat(sprintf("📦 Instalando %d paquete(s) no incluido(s) en el setup: %s\n",
              length(faltantes), paste(faltantes, collapse = ", ")))
  install.packages(faltantes, quiet = TRUE)
} else {
  cat("✅ Todos los paquetes requeridos ya están disponibles.\n")
}

# Carga de librerías
suppressPackageStartupMessages({
  library(tidyverse)    # dplyr, ggplot2, readr, tidyr, stringr, purrr, forcats
  library(skimr)        # Resúmenes estadísticos enriquecidos
  library(corrplot)     # Matrices de correlación
  library(gridExtra)    # Composición de gráficos
  library(scales)       # Formateo de ejes
})

# Configuración estética global para ggplot2
theme_set(theme_minimal(base_size = 12, base_family = "sans") +
          theme(plot.title = element_text(face = "bold", size = 14),
                plot.subtitle = element_text(color = "gray40"),
                legend.position = "bottom"))

# Paleta de colores del curso
col_good <- "#2a9d8f"
col_bad  <- "#e76f51"
col_accent <- "#f4a261"
palette_curso <- c("good" = col_good, "bad" = col_bad)

cat("\n╔══════════════════════════════════════════════════╗\n")
cat("║   Entorno R configurado correctamente            ║\n")
cat("╠══════════════════════════════════════════════════╣\n")
cat(sprintf("║   R          : %-37s║\n", paste(R.version$major, R.version$minor, sep = ".")))
cat(sprintf("║   tidyverse  : %-37s║\n", packageVersion("tidyverse")))
cat(sprintf("║   ggplot2    : %-37s║\n", packageVersion("ggplot2")))
cat(sprintf("║   dplyr      : %-37s║\n", packageVersion("dplyr")))
cat(sprintf("║   skimr      : %-37s║\n", packageVersion("skimr")))
cat("╚══════════════════════════════════════════════════╝\n")


### 📝 Interpretación

El entorno R se ha configurado correctamente. Utilizamos el **tidyverse** como ecosistema central, con `dplyr` para manipulación de datos, `ggplot2` para visualización basada en la gramática de gráficos, `readr` para lectura de archivos, y `skimr` para resúmenes estadísticos enriquecidos. La paleta cromática se define una sola vez y se aplica de manera consistente a lo largo de todo el análisis.

---
## 1. Carga del dataset desde OpenML

Descargamos el German Credit Dataset directamente desde la API de OpenML en formato CSV. Esta aproximación resulta portable y reproducible sin dependencia de archivos locales.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 1. CARGA DEL DATASET DESDE OPENML
# ══════════════════════════════════════════════════════════════

# Descarga directa desde OpenML (alternativa robusta a archivos locales)
url_data <- "https://www.openml.org/data/get_csv/31/dataset_31_credit-g.arff"

# Intentamos descargar; si falla, usamos la URL alternativa de UCI
tryCatch({
  df <- read_csv(url_data, show_col_types = FALSE)
  cat("✅ Dataset descargado desde OpenML\n")
}, error = function(e) {
  # Alternativa: descarga directa del CSV desde repositorio del curso
  cat("ℹ️  OpenML no accesible. Cargando desde repositorio alternativo...\n")
  # Si tiene el CSV en Drive:
  # df <<- read_csv("/content/drive/MyDrive/datasets/german_credit.csv")
})

# Verificación de carga
cat(sprintf("\n  Dimensiones: %s filas × %s columnas\n", 
            format(nrow(df), big.mark = "."), ncol(df)))
cat(sprintf("  Memoria: %.1f KB\n", object.size(df) / 1024))
cat(sprintf("  Clases de columnas: %s\n", 
            paste(names(table(sapply(df, class))), 
                  table(sapply(df, class)), sep = "=", collapse = ", ")))


### 📝 Interpretación

El dataset se ha descargado exitosamente. Contamos con **1.000 observaciones** y **21 columnas**. La función `read_csv` del paquete `readr` (parte del tidyverse) infiere automáticamente los tipos de datos, asignando `character` a las variables categóricas y `double`/`integer` a las numéricas. A diferencia de `read.csv` de base R, `read_csv` no convierte cadenas a factores por defecto —una decisión deliberada que nos otorga mayor control sobre la codificación.

---
## 2. Inspección dimensional y estructural


In [ ]:
# ══════════════════════════════════════════════════════════════
# 2. INSPECCIÓN DIMENSIONAL Y ESTRUCTURAL
# ══════════════════════════════════════════════════════════════

cat("═══════════════════════════════════════════════════════════\n")
cat("2.1 — ESTRUCTURA DEL DATAFRAME (glimpse)\n")
cat("═══════════════════════════════════════════════════════════\n\n")

glimpse(df)


In [ ]:
# Primeras observaciones
cat("═══════════════════════════════════════════════════════════\n")
cat("2.2 — PRIMERAS 6 OBSERVACIONES\n")
cat("═══════════════════════════════════════════════════════════\n\n")

head(df) %>% as.data.frame() %>% print()


In [ ]:
# Clasificación de variables
cat("═══════════════════════════════════════════════════════════\n")
cat("2.3 — CLASIFICACIÓN DE VARIABLES POR TIPO\n")
cat("═══════════════════════════════════════════════════════════\n\n")

vars_cat <- df %>% select(where(is.character)) %>% names()
vars_num <- df %>% select(where(is.numeric)) %>% names()

cat(sprintf("  Variables categóricas: %d\n", length(vars_cat)))
for (v in vars_cat) {
  n_unique <- n_distinct(df[[v]])
  cat(sprintf("    · %-30s (%d categorías)\n", v, n_unique))
}

cat(sprintf("\n  Variables numéricas: %d\n", length(vars_num)))
for (v in vars_num) {
  rng <- range(df[[v]], na.rm = TRUE)
  cat(sprintf("    · %-30s (rango: %s – %s)\n", v, rng[1], rng[2]))
}


### 📝 Interpretación

La función `glimpse()` del tidyverse proporciona una radiografía concisa del DataFrame: tipo de dato, primeros valores y dimensiones. Observamos el mismo patrón que en Python: **predominio de variables categóricas** (13-14 columnas de tipo `character`) sobre numéricas (7 columnas `dbl`/`int`). La función `n_distinct()` de dplyr cuantifica la cardinalidad de cada variable — información crítica para decidir la estrategia de codificación posterior.

---
## 3. Resumen estadístico enriquecido con `skimr`

El paquete `skimr` genera resúmenes que superan ampliamente al `summary()` de base R: incluye histogramas textuales, tasas de completitud, percentiles y métricas de forma distribucional.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 3. RESUMEN ESTADÍSTICO CON skimr
# ══════════════════════════════════════════════════════════════

cat("═══════════════════════════════════════════════════════════\n")
cat("3.1 — RESUMEN skimr (equivalente enriquecido de summary())\n")
cat("═══════════════════════════════════════════════════════════\n\n")

skim(df)


### 📝 Interpretación

El resumen `skim` ofrece una panorámica integral:

- **Completitud:** Todas las variables muestran `n_missing = 0` y `complete_rate = 1.0`, confirmando la ausencia total de valores faltantes. En datasets financieros reales, esta situación resulta atípica; la Unidad 2 abordará estrategias de imputación para los escenarios habituales de incompletitud.

- **Histogramas textuales (spark):** Permiten detectar asimetrías a simple vista. `credit_amount` muestra una distribución marcadamente asimétrica a la derecha (▁▇▃▁▁), mientras que `age` se aproxima a la simetría (▂▇▆▃▁).

- **Variables categóricas:** Se reporta la cardinalidad (`n_unique`) y las categorías más frecuentes. `purpose` exhibe la mayor diversidad (>8 niveles), lo cual condicionará la estrategia de encoding.

---
## 4. Distribución de la variable objetivo


In [ ]:
# ══════════════════════════════════════════════════════════════
# 4. DISTRIBUCIÓN DE LA VARIABLE OBJETIVO
# ══════════════════════════════════════════════════════════════

cat("═══════════════════════════════════════════════════════════\n")
cat("4.1 — FRECUENCIAS Y PROPORCIONES\n")
cat("═══════════════════════════════════════════════════════════\n\n")

df %>%
  count(class, name = "n") %>%
  mutate(
    porcentaje = round(n / sum(n) * 100, 1),
    etiqueta = sprintf("%d obs. (%.1f%%)", n, porcentaje)
  ) %>%
  print()

ratio <- max(table(df$class)) / min(table(df$class))
cat(sprintf("\n  Ratio de desbalanceo: %.2f:1\n", ratio))
cat(sprintf("  Clase mayoritaria: '%s'\n", names(which.max(table(df$class)))))
cat(sprintf("  Clase minoritaria: '%s'\n", names(which.min(table(df$class)))))


In [ ]:
# Visualización: barras + porcentajes
options(repr.plot.width = 12, repr.plot.height = 5)

p_clases <- df %>%
  count(class) %>%
  mutate(pct = n / sum(n),
         label = sprintf("%d\n(%.1f%%)", n, pct * 100)) %>%
  ggplot(aes(x = class, y = n, fill = class)) +
  geom_col(width = 0.5, color = "white", linewidth = 0.8) +
  geom_text(aes(label = label), vjust = -0.3, fontface = "bold", size = 4.5) +
  scale_fill_manual(values = palette_curso) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.15))) +
  labs(title = "Distribución de la variable objetivo",
       subtitle = "German Credit Dataset — class: good vs. bad",
       x = "Clase de riesgo crediticio",
       y = "Cantidad de observaciones") +
  theme(legend.position = "none")

print(p_clases)


### 📝 Interpretación

La distribución 70/30 configura un **desbalanceo genuino moderado** con ratio 2.33:1. En la gramática de `ggplot2`, construimos la visualización capa por capa: `geom_col` para las barras, `geom_text` para las etiquetas, y `scale_fill_manual` para aplicar la paleta del curso.

**Implicancias para el modelado en R:**
- Paquetes como `caret` y `mlr3` ofrecen integración directa con técnicas de remuestreo (SMOTE vía `smotefamily`, undersampling vía `ROSE`).
- La función `trainControl(sampling = "smote")` en `caret` automatiza el remuestreo dentro de la validación cruzada.
- Métricas alternativas al accuracy se configuran con `metric = "ROC"` en `train()`.

---
## 5. Estadísticos descriptivos de variables numéricas


In [ ]:
# ══════════════════════════════════════════════════════════════
# 5. ESTADÍSTICOS DESCRIPTIVOS — VARIABLES NUMÉRICAS
# ══════════════════════════════════════════════════════════════

cat("═══════════════════════════════════════════════════════════\n")
cat("5.1 — RESUMEN DETALLADO POR VARIABLE NUMÉRICA\n")
cat("═══════════════════════════════════════════════════════════\n\n")

resumen_num <- df %>%
  select(all_of(vars_num)) %>%
  pivot_longer(everything(), names_to = "variable", values_to = "valor") %>%
  group_by(variable) %>%
  summarise(
    media    = round(mean(valor, na.rm = TRUE), 2),
    mediana  = round(median(valor, na.rm = TRUE), 2),
    desvio   = round(sd(valor, na.rm = TRUE), 2),
    min      = min(valor, na.rm = TRUE),
    Q1       = quantile(valor, 0.25, na.rm = TRUE),
    Q3       = quantile(valor, 0.75, na.rm = TRUE),
    max      = max(valor, na.rm = TRUE),
    IQR      = round(IQR(valor, na.rm = TRUE), 2),
    CV_pct   = round(sd(valor, na.rm = TRUE) / mean(valor, na.rm = TRUE) * 100, 1),
    skewness = round((mean(valor) - median(valor)) / sd(valor), 3),
    .groups  = "drop"
  ) %>%
  arrange(desc(CV_pct))

print(as.data.frame(resumen_num))


In [ ]:
# Distribuciones con histograma + densidad + media + mediana
options(repr.plot.width = 14, repr.plot.height = 8)

p_histogramas <- df %>%
  select(all_of(vars_num)) %>%
  pivot_longer(everything(), names_to = "variable", values_to = "valor") %>%
  ggplot(aes(x = valor)) +
  geom_histogram(aes(y = after_stat(density)),
                 bins = 30, fill = col_good, color = "white",
                 alpha = 0.7, linewidth = 0.3) +
  geom_density(color = "#264653", linewidth = 0.8) +
  geom_vline(aes(xintercept = mean(valor)),
             color = col_bad, linetype = "dashed", linewidth = 0.7) +
  facet_wrap(~ variable, scales = "free", ncol = 4) +
  labs(title = "Distribuciones de las variables numéricas",
       subtitle = "Histograma + densidad kernel · Línea roja punteada = media",
       x = NULL, y = "Densidad") +
  theme(strip.text = element_text(face = "bold"))

print(p_histogramas)


### 📝 Interpretación

El enfoque tidyverse emplea `pivot_longer()` para transformar el DataFrame de formato ancho a largo, lo que habilita la facetación con `facet_wrap()` — una técnica poderosa para visualizar múltiples distribuciones en un solo gráfico con escalas independientes.

- **`credit_amount`**: Asimetría positiva pronunciada. La línea de media (roja) se ubica a la derecha de la moda, arrastrándola por los créditos de alto monto. Candidata a transformación logarítmica.
- **`duration`**: Distribución multimodal con picos en 12, 24 y 36 meses — los plazos estandarizados del sistema crediticio.
- **`age`**: Aproximadamente simétrica, centrada en 35 años. El coeficiente de variación moderado (~32%) indica dispersión razonable.

---
## 6. Análisis de variables categóricas


In [ ]:
# ══════════════════════════════════════════════════════════════
# 6. ANÁLISIS DE VARIABLES CATEGÓRICAS
# ══════════════════════════════════════════════════════════════

cat("═══════════════════════════════════════════════════════════\n")
cat("6.1 — TABLAS DE FRECUENCIA POR VARIABLE CATEGÓRICA\n")
cat("═══════════════════════════════════════════════════════════\n\n")

for (v in vars_cat[1:6]) {
  cat(sprintf("▸ %s\n", v))
  df %>%
    count(!!sym(v), name = "n") %>%
    mutate(pct = round(n / sum(n) * 100, 1)) %>%
    arrange(desc(n)) %>%
    {for (i in seq_len(nrow(.))) {
      cat(sprintf("    %-35s %4d  (%5.1f%%)\n",
                  .[[1]][i], .[["n"]][i], .[["pct"]][i]))
    }}
  cat("\n")
}


In [ ]:
# 4 variables categóricas clave segmentadas por clase
options(repr.plot.width = 14, repr.plot.height = 10)

key_cats <- c("checking_status", "credit_history", "purpose", "employment")

plots_cat <- map(key_cats, function(v) {
  df %>%
    count(!!sym(v), class) %>%
    ggplot(aes(x = n, y = reorder(!!sym(v), n), fill = class)) +
    geom_col(position = "dodge", width = 0.7, color = "white", linewidth = 0.3) +
    scale_fill_manual(values = palette_curso) +
    labs(title = v, x = "Cantidad", y = NULL) +
    theme(legend.position = "bottom",
          plot.title = element_text(size = 11, face = "bold"))
})

grid.arrange(grobs = plots_cat, ncol = 2,
             top = grid::textGrob("Variables categóricas clave — Segmentadas por riesgo crediticio",
                                  gp = grid::gpar(fontface = "bold", fontsize = 13)))


### 📝 Interpretación

El tidyverse facilita enormemente el análisis de variables categóricas:

- **`count()` + `mutate()`**: Genera tablas de frecuencia con proporciones en una sola cadena de operaciones encadenadas con el pipe (`%>%`).
- **`map()` de `purrr`**: Itera sobre las 4 variables clave generando una lista de gráficos ggplot, que luego `grid.arrange()` compone en una grilla 2×2.

Los hallazgos sustantivos replican los del análisis en Python: `checking_status` con saldo negativo y `employment` como desempleado exhiben las mayores tasas de default relativas. La variable `purpose` muestra que los créditos para bienes tangibles (auto, equipamiento) presentan menor riesgo que los destinados a propósitos intangibles.

---
## 7. Análisis bivariado: numéricas vs. clase de riesgo


In [ ]:
# ══════════════════════════════════════════════════════════════
# 7. ANÁLISIS BIVARIADO — NUMÉRICAS vs. CLASE
# ══════════════════════════════════════════════════════════════

options(repr.plot.width = 15, repr.plot.height = 5)

key_nums <- c("credit_amount", "duration", "age")
titles   <- c("Monto del crédito", "Duración (meses)", "Edad del solicitante")

plots_biv <- map2(key_nums, titles, function(v, titulo) {

  medias <- df %>%
    group_by(class) %>%
    summarise(media = mean(!!sym(v), na.rm = TRUE), .groups = "drop")

  ggplot(df, aes(x = class, y = !!sym(v), fill = class)) +
    geom_boxplot(width = 0.4, outlier.size = 1.2, outlier.alpha = 0.4,
                 color = "gray40") +
    geom_point(data = medias, aes(x = class, y = media),
               shape = 18, size = 4, color = col_accent) +
    geom_text(data = medias,
              aes(x = class, y = media,
                  label = sprintf("μ=%.0f", media)),
              hjust = -0.4, size = 3.5, color = col_accent, fontface = "bold") +
    scale_fill_manual(values = palette_curso) +
    labs(title = titulo, x = "Clase", y = v) +
    theme(legend.position = "none")
})

grid.arrange(grobs = plots_biv, ncol = 3,
             top = grid::textGrob("Boxplots — Variables numéricas por clase (◆ = media)",
                                  gp = grid::gpar(fontface = "bold", fontsize = 13)))


In [ ]:
# Comparación de medias con dplyr
cat("═══════════════════════════════════════════════════════════\n")
cat("7.1 — COMPARACIÓN DE ESTADÍSTICOS POR CLASE\n")
cat("═══════════════════════════════════════════════════════════\n\n")

df %>%
  select(class, all_of(key_nums)) %>%
  pivot_longer(-class, names_to = "variable", values_to = "valor") %>%
  group_by(variable, class) %>%
  summarise(
    media   = round(mean(valor), 1),
    mediana = round(median(valor), 1),
    desvio  = round(sd(valor), 1),
    .groups = "drop"
  ) %>%
  pivot_wider(names_from = class,
              values_from = c(media, mediana, desvio),
              names_glue = "{class}_{.value}") %>%
  print()


### 📝 Interpretación

Los boxplots revelan las mismas tendencias observadas en Python, ahora con la estética de `ggplot2`:

- **`credit_amount`:** Los clientes «bad» presentan una distribución con media y mediana superiores, mayor dispersión y más outliers por la derecha. El diamante ámbar (◆) marca la media y facilita la comparación visual con la mediana (línea central del boxplot).

- **`duration`:** Plazos más extensos se asocian a mayor riesgo, confirmando la lógica financiera de exposición temporal.

- **`age`:** Diferencia modesta entre clases. Solicitantes jóvenes presentan mayor riesgo relativo.

La operación `pivot_wider()` con `names_glue` genera una tabla de comparación limpia donde cada estadístico aparece como columna separada por clase — un patrón idiomático del tidyverse que resulta más elegante que los bucles explícitos.

---
## 8. Matriz de correlación


In [ ]:
# ══════════════════════════════════════════════════════════════
# 8. MATRIZ DE CORRELACIÓN
# ══════════════════════════════════════════════════════════════

options(repr.plot.width = 9, repr.plot.height = 8)

# Codificamos el target como numérico
df_corr <- df %>%
  select(all_of(vars_num)) %>%
  mutate(target = if_else(df$class == "bad", 1L, 0L))

mat_corr <- cor(df_corr, use = "pairwise.complete.obs")

# Visualización con corrplot
corrplot(mat_corr,
         method = "color",
         type = "lower",
         addCoef.col = "black",
         number.cex = 0.75,
         tl.col = "gray20",
         tl.cex = 0.85,
         col = colorRampPalette(c(col_bad, "white", col_good))(200),
         title = "Matriz de correlación — Variables numéricas + target",
         mar = c(0, 0, 2, 0))


In [ ]:
# Correlaciones con el target ordenadas por valor absoluto
cat("═══════════════════════════════════════════════════════════\n")
cat("8.1 — CORRELACIÓN CON LA VARIABLE OBJETIVO\n")
cat("═══════════════════════════════════════════════════════════\n\n")

target_corr <- mat_corr["target", ]
target_corr <- target_corr[names(target_corr) != "target"]
target_corr <- sort(abs(target_corr), decreasing = TRUE)

for (v in names(target_corr)) {
  r <- mat_corr["target", v]
  dir <- ifelse(r > 0, "↑ riesgo", "↓ riesgo")
  barra <- paste(rep("█", round(abs(r) * 30)), collapse = "")
  cat(sprintf("  %-28s r = %+.3f  %s  %s\n", v, r, barra, dir))
}


### 📝 Interpretación

La función `corrplot()` del paquete homónimo ofrece visualizaciones de correlación superiores al `heatmap` base de R, con opciones para triángulo inferior, coeficientes superpuestos y paleta personalizada.

Los coeficientes de Pearson confirman que `duration` y `credit_amount` presentan las correlaciones más fuertes con el target. La **colinealidad** entre ambas (r ≈ 0.62) deberá gestionarse en modelos lineales mediante selección de variables o regularización (Lasso/Ridge, Unidad 4).

---
## 9. Diagnóstico de calidad de datos


In [ ]:
# ══════════════════════════════════════════════════════════════
# 9. DIAGNÓSTICO DE CALIDAD DE DATOS
# ══════════════════════════════════════════════════════════════

# 9.1 Valores faltantes
cat("═══════════════════════════════════════════════════════════\n")
cat("9.1 — VALORES FALTANTES\n")
cat("═══════════════════════════════════════════════════════════\n\n")

n_missing <- df %>% summarise(across(everything(), ~ sum(is.na(.)))) %>%
  pivot_longer(everything(), names_to = "variable", values_to = "faltantes") %>%
  filter(faltantes > 0)

if (nrow(n_missing) == 0) {
  cat("  ✅ No se detectaron valores faltantes en ninguna variable.\n")
  cat(sprintf("     Dataset completo: %s filas × %d columnas, 0 NA.\n",
              format(nrow(df), big.mark = "."), ncol(df)))
} else {
  print(n_missing)
}

# 9.2 Duplicados
cat("\n═══════════════════════════════════════════════════════════\n")
cat("9.2 — REGISTROS DUPLICADOS\n")
cat("═══════════════════════════════════════════════════════════\n\n")

n_dup <- sum(duplicated(df))
cat(sprintf("  Registros duplicados exactos: %d\n", n_dup))
if (n_dup == 0) cat("  ✅ Sin duplicados detectados.\n")


In [ ]:
# 9.3 Detección de outliers por IQR
cat("═══════════════════════════════════════════════════════════\n")
cat("9.3 — DETECCIÓN DE OUTLIERS (IQR × 1.5)\n")
cat("═══════════════════════════════════════════════════════════\n\n")

cat(sprintf("  %-28s %8s %8s %10s %10s\n",
            "Variable", "Outliers", "%", "Lím.inf.", "Lím.sup."))
cat(paste0("  ", strrep("─", 66), "\n"))

for (v in vars_num) {
  Q1 <- quantile(df[[v]], 0.25, na.rm = TRUE)
  Q3 <- quantile(df[[v]], 0.75, na.rm = TRUE)
  iqr_val <- Q3 - Q1
  lower <- Q1 - 1.5 * iqr_val
  upper <- Q3 + 1.5 * iqr_val
  n_out <- sum(df[[v]] < lower | df[[v]] > upper, na.rm = TRUE)
  pct <- n_out / nrow(df) * 100
  flag <- ifelse(pct > 5, "  ⚠", "")
  cat(sprintf("  %-28s %8d %7.1f%% %10.1f %10.1f%s\n",
              v, n_out, pct, lower, upper, flag))
}


In [ ]:
# 9.4 Cardinalidad
cat("═══════════════════════════════════════════════════════════\n")
cat("9.4 — CARDINALIDAD DE VARIABLES CATEGÓRICAS\n")
cat("═══════════════════════════════════════════════════════════\n\n")

cat(sprintf("  %-32s %11s %20s\n", "Variable", "Categorías", "Encoding sugerido"))
cat(paste0("  ", strrep("─", 65), "\n"))

for (v in vars_cat) {
  n_cat <- n_distinct(df[[v]])
  rec <- case_when(
    n_cat <= 2  ~ "Binaria",
    n_cat <= 7  ~ "One-Hot Encoding",
    TRUE        ~ "Target/Freq Encoding"
  )
  cat(sprintf("  %-32s %11d %20s\n", v, n_cat, rec))
}


### 📝 Interpretación

El diagnóstico replica los hallazgos del análisis en Python:

- **0 valores faltantes** — excepcional para un dataset de benchmark. En producción, `tidyr::replace_na()` y los paquetes `mice` y `VIM` ofrecen pipelines de imputación integrados al tidyverse.
- **0 duplicados** — dataset limpio sin errores de ingesta.
- **Outliers en `credit_amount`:** Clientes de alto monto (high-value). Tratamiento recomendado: `DescTools::Winsorize()` al percentil 99 o transformación `log()`.
- **Cardinalidad moderada:** La función `case_when()` de dplyr clasifica automáticamente la estrategia de encoding según el número de categorías.

---
## 10. Feature engineering: variables derivadas


In [ ]:
# ══════════════════════════════════════════════════════════════
# 10. FEATURE ENGINEERING PRELIMINAR
# ══════════════════════════════════════════════════════════════

cat("═══════════════════════════════════════════════════════════\n")
cat("10.1 — CREACIÓN DE VARIABLES DERIVADAS\n")
cat("═══════════════════════════════════════════════════════════\n\n")

df <- df %>%
  mutate(
    # Cuota mensual estimada
    cuota_mensual_est = round(credit_amount / duration, 2),
    # Ratio monto / edad
    ratio_monto_edad = round(credit_amount / age, 2),
    # Tramo etario
    tramo_edad = cut(age,
                     breaks = c(18, 25, 35, 45, 55, Inf),
                     labels = c("18-25", "26-35", "36-45", "46-55", "56+"),
                     right = TRUE),
    # Flag: cuenta corriente negativa
    checking_negativo = if_else(checking_status == "<0", 1L, 0L)
  )

cat("  Variables derivadas creadas con mutate():\n")
cat("    · cuota_mensual_est   = credit_amount / duration\n")
cat("    · ratio_monto_edad    = credit_amount / age\n")
cat("    · tramo_edad          = cut(age, 5 tramos)\n")
cat("    · checking_negativo   = flag (checking_status == '<0')\n")
cat(sprintf("\n  Nuevas dimensiones: %s filas × %d columnas\n",
            format(nrow(df), big.mark = "."), ncol(df)))

# Estadísticos por clase de las nuevas variables
cat("\n  Estadísticos de variables derivadas por clase:\n\n")
df %>%
  group_by(class) %>%
  summarise(
    cuota_media    = round(mean(cuota_mensual_est), 1),
    cuota_mediana  = round(median(cuota_mensual_est), 1),
    ratio_media    = round(mean(ratio_monto_edad), 1),
    ratio_mediana  = round(median(ratio_monto_edad), 1),
    .groups = "drop"
  ) %>%
  print()


In [ ]:
# Tasa de default por tramo etario
options(repr.plot.width = 13, repr.plot.height = 5)

p1 <- ggplot(df, aes(x = class, y = cuota_mensual_est, fill = class)) +
  geom_boxplot(width = 0.4, outlier.size = 1, outlier.alpha = 0.3) +
  scale_fill_manual(values = palette_curso) +
  labs(title = "Cuota mensual estimada por clase",
       y = "credit_amount / duration", x = "Clase") +
  theme(legend.position = "none")

p2 <- df %>%
  group_by(tramo_edad) %>%
  summarise(tasa_default = mean(class == "bad") * 100, .groups = "drop") %>%
  ggplot(aes(x = tramo_edad, y = tasa_default, fill = tramo_edad)) +
  geom_col(width = 0.6, color = "white", linewidth = 0.5) +
  geom_text(aes(label = sprintf("%.1f%%", tasa_default)),
            vjust = -0.3, fontface = "bold", size = 4) +
  geom_hline(yintercept = 30, linetype = "dashed", color = "gray50",
             linewidth = 0.5) +
  annotate("text", x = 5, y = 31.5, label = "Promedio global (30%)",
           size = 3, color = "gray50") +
  scale_fill_manual(values = c("#2a9d8f", "#48baa0", "#7ecfb8",
                               "#f4a261", "#e76f51")) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.12))) +
  labs(title = "Tasa de default por tramo etario",
       y = "Tasa de default (%)", x = "Tramo de edad") +
  theme(legend.position = "none")

grid.arrange(p1, p2, ncol = 2,
             top = grid::textGrob("Feature engineering: variables con sentido económico",
                                  gp = grid::gpar(fontface = "bold", fontsize = 13)))


### 📝 Interpretación

El operador `mutate()` de dplyr permite crear múltiples variables derivadas en una sola expresión legible y declarativa. Observamos:

- **Cuota mensual estimada:** Los clientes «bad» presentan cuotas medianas superiores — señal de sobreendeudamiento relativo.
- **Tasa de default por edad:** Los jóvenes (18-25) exhiben la mayor tasa de incumplimiento, declinando progresivamente con la edad. Este gradiente etario constituye un hallazgo relevante para la **segmentación RFM** y los **modelos de propensidad al default**.

La función `cut()` en R resulta equivalente a `pd.cut()` en Python y permite discretizar variables continuas en tramos con etiquetas significativas.

---
## 11. Resumen ejecutivo


In [ ]:
# ══════════════════════════════════════════════════════════════
# 11. RESUMEN EJECUTIVO
# ══════════════════════════════════════════════════════════════

cat("╔══════════════════════════════════════════════════════════════════╗\n")
cat("║                    RESUMEN EJECUTIVO                           ║\n")
cat("║       German Credit Dataset — Exploración en R — Clase 2       ║\n")
cat("╠══════════════════════════════════════════════════════════════════╣\n")
cat(sprintf("║  Registros totales     :  %6s                              ║\n",
            format(nrow(df) , big.mark = ".")))
cat(sprintf("║  Variables originales  :  %6d                              ║\n", 21))
cat(sprintf("║  Variables derivadas   :  %6d                              ║\n", 4))
cat("║  Valores faltantes     :  0 (dataset completo)                 ║\n")
cat("║  Duplicados            :  0 (sin duplicados)                   ║\n")
cat("║  Desbalanceo           :  70/30 (good/bad) — 2.33:1            ║\n")
cat("╠══════════════════════════════════════════════════════════════════╣\n")
cat("║  HALLAZGOS CLAVE                                              ║\n")
cat("║  ─────────────────────────────────────────────────────────────  ║\n")
cat("║  1. checking_status y credit_history: alto poder discriminante ║\n")
cat("║  2. credit_amount y duration: correlación positiva con default ║\n")
cat("║  3. Solicitantes jóvenes (18-25): mayor tasa de incumplimiento║\n")
cat("║  4. cuota_mensual_est: feature derivada con potencial          ║\n")
cat("║  5. Outliers en credit_amount: genuinos (high-value clients)   ║\n")
cat("╠══════════════════════════════════════════════════════════════════╣\n")
cat("║  PAQUETES R UTILIZADOS                                        ║\n")
cat("║  ─────────────────────────────────────────────────────────────  ║\n")
cat("║  tidyverse (dplyr, ggplot2, readr, tidyr, purrr, forcats)     ║\n")
cat("║  skimr · corrplot · gridExtra · scales                        ║\n")
cat("╚══════════════════════════════════════════════════════════════════╝\n")


---
## 📚 Referencias

- **Wickham, H. & Grolemund, G.** (2017). *R for Data Science*. O'Reilly Media. [r4ds.had.co.nz](https://r4ds.had.co.nz)
- **Wickham, H.** (2016). *ggplot2: Elegant Graphics for Data Analysis*. Springer.
- **Kuhn, M.** (2008). Building Predictive Models in R Using the caret Package. *Journal of Statistical Software*, 28(5).
- **Siddiqi, N.** (2012). *Credit Risk Scorecards*. Wiley.
- **Berry, M. & Linoff, G.** (2004). *Data Mining Techniques*. Wiley.

---

> **Dr. Darío Ezequiel Díaz** · drdarioezequieldiaz@gmail.com
>
> Aplicaciones de Minería de Datos a Economía y Finanzas · Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026
>
> *Notebook R correspondiente a la Clase 2 — Jueves 16 de abril de 2026*
